In [2]:
import torch
import torch.nn as nn

class AutoregressiveSpinModel(nn.Module):
    def __init__(self, num_spins, hidden_dim=64):
        super().__init__()
        self.num_spins = num_spins
        self.net = nn.Sequential(
            nn.Linear(num_spins, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_spins * 2)  # 输出每个自旋的logit
        )
    
    def forward(self, spins):
        # spins: (batch_size, num_spins), 取值{-1, 1}
        logits = self.net(spins).view(-1, self.num_spins, 2)
        # 计算条件概率 P(σ_i | σ_<i)
        cond_probs = torch.softmax(logits, dim=-1)
        return cond_probs

    def log_prob(self, spins):
        cond_probs = self.forward(spins)
        # 选择对应自旋值的概率
        selected_probs = cond_probs.gather(-1, (spins.unsqueeze(-1) + 1) // 2)
        return torch.log(selected_probs).sum(dim=1)  # 联合对数概率